# Experiment 2, step 1: which parts of the prompt drive the edit?

Reads `results/<tag>/` from `scripts/02_summarise.py`. Measure: how much less likely the model's own response becomes (nats per response token) when one prompt span is deleted, and when the parent's score is swapped. 360 mutation queries from experiment 1 `main`, 120 per operator. CIs: 95% bootstrap over LLaMEA runs.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

MI_ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(MI_ROOT))
from common.paths import results_dir
from common.plotting import INK, NEUTRAL, SERIES, setup

setup()
EXP, TAG = "exp02_feedback_attribution", "main"
res = results_dir(EXP, TAG)
drop = pd.read_csv(res / "span_drop.csv")
grad = pd.read_csv(res / "span_gradients.csv")
by_parent = pd.read_csv(res / "span_drop_by_parent.csv")
cf = pd.read_csv(res / "score_counterfactual.csv")
sim = pd.read_csv(res / "parent_child_similarity.csv")
drop.pivot(index="span", columns="operator", values="mean").round(4)

In [ ]:
cf.round(4)

In [ ]:
spans = ["parent_code", "format", "population", "task", "operator", "parent_description", "feedback"]
labels = {"parent_code": "parent code", "format": "output format", "population": "population summary",
          "task": "task description", "operator": "operator instruction",
          "parent_description": "parent description", "feedback": "feedback (score / error)"}
ops = [("refine", SERIES[0]), ("new", SERIES[1]), ("simplify", SERIES[2])]
fig, axes = plt.subplots(1, 3, figsize=(14, 4), gridspec_kw={"width_ratios": [1.4, 1, 0.8]})
ax = axes[0]
for k, (op, c) in enumerate(ops):
    d = drop[drop["operator"] == op].set_index("span").loc[spans]
    y = np.arange(len(spans)) + (k - 1) * 0.22
    ax.errorbar(d["mean"], y, xerr=[d["mean"] - d["ci_low"], d["ci_high"] - d["mean"]],
                fmt="o", color=c, ms=5, lw=1.5, label=op)
ax.set_xscale("log"); ax.set_xlim(5e-4, 1)
ax.set_yticks(range(len(spans))); ax.set_yticklabels([labels[s] for s in spans])
ax.invert_yaxis(); ax.grid(axis="y", visible=False)
ax.set_xlabel("drop in response log-prob when deleted (nats/token, log scale)")
ax.set_title("Causal dependence of the response on each span", loc="left", color=INK, fontsize=10)
ax.legend(title="operator", fontsize=8, title_fontsize=8, loc="lower right")
ax = axes[1]
for k, (failed, c, lbl) in enumerate([(False, SERIES[0], "parent works (feedback = score)"),
                                      (True, SERIES[3], "parent failed (feedback = error)")]):
    d = by_parent[by_parent["parent_failed"] == failed].set_index("span").loc[spans]
    y = np.arange(len(spans)) + (k - 0.5) * 0.25
    ax.errorbar(d["mean"], y, xerr=[d["mean"] - d["ci_low"], d["ci_high"] - d["mean"]],
                fmt="o", color=c, ms=5, lw=1.5, label=lbl)
ax.set_xscale("log"); ax.set_xlim(5e-4, 1)
ax.set_yticks(range(len(spans))); ax.set_yticklabels([])
ax.invert_yaxis(); ax.grid(axis="y", visible=False)
ax.set_xlabel("drop (nats/token, log scale)")
ax.set_title("Feedback matters only when it is an error", loc="left", color=INK, fontsize=10)
ax.legend(fontsize=8, loc="upper left")
ax = axes[2]
for k, (op, c) in enumerate(ops):
    v = sim.loc[sim["operator"] == op, "sim"]
    parts = ax.violinplot(v, positions=[k], showextrema=False, widths=0.7)
    for b in parts["bodies"]:
        b.set_facecolor(c); b.set_alpha(0.35); b.set_edgecolor("none")
    ax.plot([k - 0.2, k + 0.2], [v.median()] * 2, color=INK, lw=2)
ax.set_xticks(range(3)); ax.set_xticklabels([o for o, _ in ops])
ax.set_ylim(0, 1); ax.set_ylabel("code similarity child vs parent")
ax.set_title("'new' children are rewrites", loc="left", color=INK, fontsize=10)
fig.tight_layout()
fig.savefig(res / "span_attribution.png", dpi=150)

In [ ]:
grad.pivot(index="span", columns="operator", values="grad_share").round(3)